Joey Cahill

In [ ]:
import pandas as pd
import geopandas as gpd
from shapely.geometry import Point
from IPython.display import display, clear_output
import ipywidgets as widgets

from ipyleaflet import (
    Map, Marker, Circle, GeoJSON, LayersControl,
    WidgetControl, MarkerCluster
)


# Load pharmacy data

pharm_fp = r"C:\Users\jcahi\Box\DAIR_SA\Data Sets\Pharmacy Data\Cleaned Datasets\PHARMACIES_places_full_results.csv"

pharm = pd.read_csv(pharm_fp)
pharm.columns = pharm.columns.str.strip().str.lower()

pharm["lat"] = pd.to_numeric(pharm["lat"], errors="coerce")
pharm["lng"] = pd.to_numeric(pharm["lng"], errors="coerce")

pharm = pharm.dropna(subset=["lat", "lng"]).copy()

# Optional QA filter
if "api_status" in pharm.columns:
    pharm = pharm[pharm["api_status"].astype(str).str.upper() == "OK"].copy()

# South Africa bounding sanity check
pharm = pharm[
    pharm["lat"].between(-36, -21) &
    pharm["lng"].between(16, 34)
].copy()


# Build GeoDataFrames

pharm_gdf = gpd.GeoDataFrame(
    pharm,
    geometry=gpd.points_from_xy(pharm["lng"], pharm["lat"]),
    crs="EPSG:4326"
)

# For buffering/distance: use projected CRS in meters
# EPSG:32735 = WGS84 / UTM zone 35S
# This is not perfect for all of SA, but is often fine for practical distance work.
pharm_proj = pharm_gdf.to_crs(epsg=32735)

# Widgets

radius_slider = widgets.IntSlider(
    value=5000,
    min=500,
    max=50000,
    step=500,
    description="Radius (m):",
    continuous_update=False,
    style={"description_width": "initial"},
    layout=widgets.Layout(width="500px")
)

results_out = widgets.Output()
info_out = widgets.Output()

# Create map

m = Map(
    center=(-29.0, 24.0),
    zoom=5,
    scroll_wheel_zoom=True
)

m.add_control(LayersControl())

# Initial pharmacy markers (clustered)
all_markers = [
    Marker(location=(row["lat"], row["lng"]), draggable=False, title=str(row.get("practice_name", "")))
    for _, row in pharm.iterrows()
]

cluster = MarkerCluster(markers=all_markers)
m.add_layer(cluster)

# State holders
home_marker = None
buffer_circle = None
selected_layer = None
clicked_latlon = None

# Helper function

def update_selection(*args):
    global selected_layer, buffer_circle, clicked_latlon

    if clicked_latlon is None:
        with info_out:
            clear_output()
            print("Click on the map to set a home location.")
        return

    lat, lon = clicked_latlon
    radius_m = radius_slider.value

    # Remove old selected layer / circle
    if buffer_circle in m.layers:
        m.remove(buffer_circle)
    if selected_layer in m.layers:
        m.remove(selected_layer)

    # Draw approximate circle on map for visualization
    buffer_circle = Circle(
        location=(lat, lon),
        radius=radius_m,
        color="blue",
        fill_color="blue",
        fill_opacity=0.15
    )
    m.add_layer(buffer_circle)

    # Spatial selection in projected CRS
    home_gdf = gpd.GeoDataFrame(
        {"id": [1]},
        geometry=[Point(lon, lat)],
        crs="EPSG:4326"
    ).to_crs(pharm_proj.crs)

    home_point_proj = home_gdf.geometry.iloc[0]
    buffer_geom = home_point_proj.buffer(radius_m)

    selected = pharm_proj[pharm_proj.geometry.within(buffer_geom)].copy()
    selected_wgs84 = selected.to_crs(epsg=4326)

    # Show selected pharmacies as GeoJSON
    if len(selected_wgs84) > 0:
        selected_layer = GeoJSON(
            data=selected_wgs84.__geo_interface__,
            style={
                "color": "red",
                "weight": 2,
                "fillColor": "red",
                "fillOpacity": 0.7
            },
            point_style={
                "radius": 6,
                "color": "red",
                "fillColor": "red",
                "fillOpacity": 0.8
            }
        )
        m.add_layer(selected_layer)

    # Update info text
    with info_out:
        clear_output()
        print(f"Home point: ({lat:.5f}, {lon:.5f})")
        print(f"Radius: {radius_m:,} meters")
        print(f"Pharmacies found: {len(selected_wgs84)}")

    # Update results table
    with results_out:
        clear_output()
        if len(selected_wgs84) == 0:
            print("No pharmacies found within the selected radius.")
        else:
            cols = [c for c in [
                "practice_name", "matched_name", "address", "matched_address",
                "city", "province", "phone"
            ] if c in selected_wgs84.columns]

            display(
                selected_wgs84[cols]
                .reset_index(drop=True)
                .rename(columns={
                    "practice_name": "Practice Name",
                    "matched_name": "Matched Name",
                    "address": "Input Address",
                    "matched_address": "Matched Address",
                    "city": "City",
                    "province": "Province",
                    "phone": "Phone"
                })
            )


# Click handler

def handle_map_click(**kwargs):
    global home_marker, clicked_latlon

    if kwargs.get("type") == "click":
        lat, lon = kwargs.get("coordinates")
        clicked_latlon = (lat, lon)

        # Remove old marker
        if home_marker in m.layers:
            m.remove(home_marker)

        home_marker = Marker(location=(lat, lon), draggable=False, title="Home")
        m.add_layer(home_marker)

        update_selection()

m.on_interaction(handle_map_click)
radius_slider.observe(update_selection, names="value")


# Layout

controls_box = widgets.VBox([radius_slider, info_out])

display(widgets.VBox([
    controls_box,
    m,
    widgets.HTML("<h4>Pharmacies within selected radius</h4>"),
    results_out
]))